In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import accelerate

/home/gautamk/dev/pptgenerator/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    device_map="auto",
    quantization_config=bnb_config,
    dtype=torch.float16
)

Loading weights: 100%|██████████| 338/338 [00:01<00:00, 322.69it/s, Materializing param=model.norm.weight]                              


In [30]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Explain transformers simply."}
]
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
def chat(model, tokenizer, message, max_length=300):
    prompt = tokenizer.apply_chat_template(
        messages,
    tokenize=False,
    add_generation_prompt = True
    )
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens = max_length,
        temperature=0.3,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.1,
        eos_token_id=tokenizer.eos_token_id
    )
    generated_tokens = output[0][inputs["input_ids"].shape[-1]:]

    reply = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
    )
    print(reply)


In [26]:
print(messages)

[{'role': 'system', 'content': 'You are a helpful assistant.'}, {'role': 'user', 'content': 'Explain transformers simply.'}, {'role': 'user', 'content': 'Explain transformers in simple terms.'}, {'role': 'user', 'content': 'Explain transformers in simple terms. I am talking about transformers in ml btw'}, {'role': 'user', 'content': 'Do you remember my last 2 messages?'}]


In [29]:
messages.append(
    {"role": "user", "content": "Hi"}
)
chat(model, tokenizer, messages)


Sure, let me explain transformers in simple terms.

### What Are Transformers?
Transformers are like magical machines that help computers understand and process information better. They're used in machine learning (ML) models to make predictions or decisions based on data.

### How Do Transformers Work?
Imagine you have a big box of toys. To play with these toys, you need to know how they work and what each toy does. Transformers do something similar but for computer programs.

1. **Input**: You give your program some input data, like instructions or questions.
2. **Hidden Layers**: Inside this magical transformer, there are hidden layers. These layers are like secret compartments where the magic happens.
3. **Output**: The final result comes out as an output, which is usually a prediction or answer based on the input.

### Why Are Transformers Important?
- **Efficiency**: Transformers can handle large amounts of data quickly because they break down complex tasks into smaller parts.
- 